In [5]:
# 1. Import Libraries
import pandas as pd
import numpy as np
import ast
from difflib import get_close_matches
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



In [6]:
# 2. Load the New TMDB Movie Dataset
df = pd.read_csv("data/tmdb_movie_dataset.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df[["tmdbId", "title", "release_date", "vote_average"]].head(3))


Dataset shape: (4602, 21)
Columns: ['budget', 'genres', 'homepage', 'tmdbId', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'ratingId']


,tmdbId,title,release_date,vote_average
0,5,Four Rooms,1995-12-09,6.5
1,11,Star Wars,1977-05-25,8.1
2,12,Finding Nemo,2003-05-30,7.6


In [25]:
# 3. Data Pre-processing and Feature Extraction

def extract_names(text):
    try:
        items = ast.literal_eval(text)
        if isinstance(items, list):
            return " ".join(
                str(item.get("name", "")) 
                for item in items
                if isinstance(item, dict)
            )
    except (ValueError, SyntaxError, TypeError):
        pass
    return ""

# Convert JSON-like genre and keyword columns into text features.
df["genres_clean"] = df["genres"].fillna("").apply(extract_names)
df["keywords_clean"] = df["keywords"].fillna("").apply(extract_names)
df["overview"] = df["overview"].fillna("")
df["year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year.astype("Int64")

# Combine movie attributes into one content field.
df["content"] = (
    df["genres_clean"] + " " +
    df["keywords_clean"] + " " +
    df["overview"]
).str.strip()

# Remove rows without a movie title or useful content.
df = df[df["title"].notna()].copy()
df["content"] = df["content"].fillna("")

display(df[["title", "genres_clean", "keywords_clean", "overview"]].head(3))


,title,genres_clean,keywords_clean,overview
0,Four Rooms,Crime Comedy,hotel new year's eve witch bet hotel room sper...,It's Ted the Bellhop's first night on the job....
1,Star Wars,Adventure Action Science Fiction,android galaxy hermit death star lightsaber je...,Princess Leia is captured and held hostage by ...
2,Finding Nemo,Animation Family,father son relationship harbor underwater fish...,"Nemo, an adventurous young clownfish, is unexp..."


In [26]:
# 4. Create TF-IDF Matrix and Cosine Similarity

tfidf = TfidfVectorizer(
    stop_words="english",
)

tfidf_matrix = tfidf.fit_transform(df["content"])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Map each movie title to its row index.
# Duplicate titles are handled by keeping the first occurrence.
indices = pd.Series(
    df.index,
    index=df["title"].str.strip().str.lower()
).drop_duplicates()

print("TF-IDF Matrix size:", tfidf_matrix.shape)
print("Cosine Similarity Matrix size:", cosine_sim.shape)


TF-IDF Matrix size: (4602, 22383)
Cosine Similarity Matrix size: (4602, 4602)


In [27]:
# 5. Content-Based Recommendation Function

def find_movie_title(title):
    """Find the dataset title matching the user's input."""
    query = str(title).strip().lower()

    if not query:
        return None

    # Exact match
    if query in indices.index:
        return df.loc[indices[query], "title"]

    # Case-insensitive partial match
    partial_matches = [
        original_title for original_title in df["title"].dropna().unique()
        if query in str(original_title).lower()
    ]

    if partial_matches:
        return partial_matches[0]

    # Fuzzy match for small spelling differences
    title_map = {
        str(t).lower(): str(t)
        for t in df["title"].dropna().unique()
    }

    close = get_close_matches(
        query,
        list(title_map.keys()),
        n=1,
        cutoff=0.65
    )

    if close:
        return title_map[close[0]]

    return None


def recommend(title, top_n=10):
    """Return top-N movies similar to the input movie."""
    matched_title = find_movie_title(title)

    if matched_title is None:
        return None

    idx = indices[matched_title.lower()]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )[1:top_n + 1]

    movie_indices = [i for i, _ in sim_scores]
    scores = [round(score, 4) for _, score in sim_scores]

    result = df[
        ["title", "year", "genres_clean", "keywords_clean"]
    ].iloc[movie_indices].copy()

    result.insert(0, "rank", range(1, len(result) + 1))
    result["similarity_score"] = scores

    result["keywords_clean"] = result["keywords_clean"].apply(
        lambda x: ", ".join(x.split()[:5]) if x else ""
    )

    result = result.rename(columns={
        "title": "movie_title",
        "genres_clean": "genres",
        "keywords_clean": "keywords"
    })

    return result.reset_index(drop=True)

In [28]:
# 6. Test the Recommendation Function

recommend("Avatar", top_n=10)


,rank,movie_title,year,genres,keywords,similarity_score
0,1,Mission to Mars,2000,Science Fiction,"mars, spacecraft, space, travel, alien",0.3055
1,2,Aliens,1986,Horror Action Thriller Science Fiction,"android, extraterrestrial, technology, space, ...",0.2876
2,3,Moonraker,1979,Action Adventure Thriller Science Fiction,"venice, mass, murder, space, marine",0.2824
3,4,Alien³,1992,Science Fiction Action Horror,"prison, android, spacecraft, space, marine",0.2766
4,5,Spaceballs,1987,Comedy Science Fiction,"android, lasergun, swordplay, temple, space",0.2561
5,6,Lifeforce,1985,Fantasy Horror Science Fiction Thriller,"space, marine, vampire, flying, saucer",0.2549
6,7,Treasure Planet,2002,Adventure Animation Family Fantasy Science Fic...,"cyborg, based, on, novel, space",0.2532
7,8,Lockout,2012,Action Thriller Science Fiction,"usa, president, anti, hero, dementia",0.2523
8,9,Alien,1979,Horror Action Thriller Science Fiction,"android, countdown, space, marine, space",0.2457
9,10,Planet of the Apes,2001,Thriller Science Fiction Action Adventure,"gorilla, space, marine, space, suit",0.2453


In [29]:
# 7. User Input - Enter a Movie Title to Get Recommendations

favorite_movie = input(
    "Please enter a movie title (e.g., Avatar): "
).strip()

matched_title = find_movie_title(favorite_movie)

if matched_title is None:
    print("\nMovie not found in the dataset.")
    print("Please try another movie title.")
else:
    print(f"\nSelected Movie: {matched_title}")
    print("\nTop 10 Content-Based Recommendations:")
    display(recommend(matched_title, top_n=10))



Selected Movie: Avatar

Top 10 Content-Based Recommendations:


,rank,movie_title,year,genres,keywords,similarity_score
0,1,Mission to Mars,2000,Science Fiction,"mars, spacecraft, space, travel, alien",0.3055
1,2,Aliens,1986,Horror Action Thriller Science Fiction,"android, extraterrestrial, technology, space, ...",0.2876
2,3,Moonraker,1979,Action Adventure Thriller Science Fiction,"venice, mass, murder, space, marine",0.2824
3,4,Alien³,1992,Science Fiction Action Horror,"prison, android, spacecraft, space, marine",0.2766
4,5,Spaceballs,1987,Comedy Science Fiction,"android, lasergun, swordplay, temple, space",0.2561
5,6,Lifeforce,1985,Fantasy Horror Science Fiction Thriller,"space, marine, vampire, flying, saucer",0.2549
6,7,Treasure Planet,2002,Adventure Animation Family Fantasy Science Fic...,"cyborg, based, on, novel, space",0.2532
7,8,Lockout,2012,Action Thriller Science Fiction,"usa, president, anti, hero, dementia",0.2523
8,9,Alien,1979,Horror Action Thriller Science Fiction,"android, countdown, space, marine, space",0.2457
9,10,Planet of the Apes,2001,Thriller Science Fiction Action Adventure,"gorilla, space, marine, space, suit",0.2453


In [30]:
# 8. Evaluation: Genre Precision@10

def genre_precision_at_k(title, k=10):
    """Share of top-k recommendations that share at least one genre."""
    matched_title = find_movie_title(title)

    if matched_title is None:
        return None

    selected_genres = set(
        df.loc[indices[matched_title.lower()], "genres_clean"].split()
    ) - {""}

    if not selected_genres:
        return None

    recommended = recommend(matched_title, k)

    if recommended is None or recommended.empty:
        return None

    matches = recommended["genres"].apply(
        lambda genres: bool(
            selected_genres & (set(str(genres).split()) - {""})
        )
    )

    return matches.mean()


import random
random.seed(42)

valid_titles = (
    df[df["genres_clean"].str.strip() != ""]["title"]
    .drop_duplicates()
    .tolist()
)

sample_size = min(100, len(valid_titles))
sample_titles = random.sample(valid_titles, sample_size)

precisions = []
for title in sample_titles:
    precision = genre_precision_at_k(title, k=10)
    if precision is not None:
        precisions.append(precision)

if precisions:
    avg_precision = sum(precisions) / len(precisions)
    print(
        f"Average Genre Precision@10 "
        f"(n={len(precisions)}): {avg_precision:.4f}"
    )
else:
    print("No valid titles were available for evaluation.")


Average Genre Precision@10 (n=100): 0.7920
